## Importación de librerías

In [2]:
import sys
import gurobipy as gp
from gurobipy import GRB
import time
import csv
import math
import random

print('Python  :', sys.version.split()[0])
print('Gurobi  :', '.'.join(map(str, gp.gurobi.version())))

Python  : 3.12.10
Gurobi  : 13.0.3


### 1. Generación de Datos (Mock Data para probar)

In [3]:
# Parámetros básicos
n_clientes = 15
N_C = list(range(1, n_clientes + 1))
N = list(range(n_clientes + 2)) # 0 es inicio, 16 es fin
D = [1, 2, 3]
R = [1, 2, 3]
N_TW = [1, 3, 5, 8, 12] 

# Generar coordenadas aleatorias (0 a 100) para el depósito y clientes
random.seed(42)
coordenadas = {i: (random.randint(0, 100), random.randint(0, 100)) for i in N}

td = {}
tt = {}
q = {}
st = {}

# Calcular matriz de distancias euclidianas y tiempos de viaje (asumimos velocidad 1)
for i in N:
    for j in N:
        if i != j:
            dist = math.sqrt((coordenadas[i][0] - coordenadas[j][0])**2 + (coordenadas[i][1] - coordenadas[j][1])**2)
            td[i, j] = dist
            tt[i, j] = dist # Tiempo proporcional a la distancia

# Generar demandas y tiempos de servicio
for i in N_C:
    base_demand = random.randint(5, 20)
    for t in D:
        # Variación de demanda por día para forzar diferencias en las rutas
        q[i, t] = base_demand * random.choice([0.8, 1.0, 1.2]) 
        st[i, t] = 10 # 10 minutos de servicio en cada local

# Estructurar el diccionario final
data_instancia = {
    'N_C': N_C, 'N': N, 'N_TW': N_TW, 'R': R, 'D': D,
    'td': td, 'tt': tt, 'q': q, 'st': st,
    'a': {1: 10, 3: 20, 5: 30, 8: 15, 12: 40}, 
    'b': {1: 150, 3: 160, 5: 190, 8: 145, 12: 180}, # Ventanas holgadas para evitar infactibilidad inicial
    'a_W': 0, 'b_W': 1000, 
    'Q': 120, # Capacidad ajustada para que las 3 rutas deban usarse
    'M': 5000 
}
print("Datos generados correctamente. Matrices listas.")

Datos generados correctamente. Matrices listas.


### 2. Definición del Framework Matemático

In [6]:
class ConsistentVRPFramework:
    def __init__(self, data):
        self.data = data
        self.model = None
        self.X = {}
        self.Z = {}
        self.W = {}
        self.dW = {}
        self.rW = {}
        
    def build_model(self):
        self.model = gp.Model("CVRP_Consistent_Panaderia")
        
        N_C, N, N_TW, R, D = self.data['N_C'], self.data['N'], self.data['N_TW'], self.data['R'], self.data['D']
        td, tt, q, st = self.data['td'], self.data['tt'], self.data['q'], self.data['st']
        a, b, a_W, b_W, Q, M = self.data['a'], self.data['b'], self.data['a_W'], self.data['b_W'], self.data['Q'], self.data['M']
        
        A = [(i, j) for i in N for j in N if i != j and j != 0 and i != max(N)]

        self.X = self.model.addVars(A, D, R, vtype=GRB.BINARY, name="X")
        self.Z = self.model.addVars(N_C, R, vtype=GRB.BINARY, name="Z")
        self.W = self.model.addVars(N_C, D, vtype=GRB.CONTINUOUS, lb=0.0, name="W")
        self.dW = self.model.addVars(R, vtype=GRB.CONTINUOUS, lb=0.0, name="dW")
        self.rW = self.model.addVars(R, D, vtype=GRB.CONTINUOUS, lb=0.0, name="rW")

        self.model.setObjective(gp.quicksum(td[i, j] * self.X[i, j, t, r] for (i, j) in A for t in D for r in R), GRB.MINIMIZE)

        for i in N_C:
            self.model.addConstr(gp.quicksum(self.Z[i, r] for r in R) == 1, name=f"AsignacionUnica_{i}")

        for j in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(gp.quicksum(self.X[i, j, t, r] for i in N if (i, j) in A) == self.Z[j, r], name=f"FlujoEntrada_{j}_{r}_{t}")
        for i in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(gp.quicksum(self.X[i, j, t, r] for j in N if (i, j) in A) == self.Z[i, r], name=f"FlujoSalida_{i}_{r}_{t}")

        for r in R:
            for t in D:
                self.model.addConstr(gp.quicksum(self.X[0, j, t, r] for j in N if (0, j) in A) == 1, name=f"SalidaDep_{r}_{t}")

        for r in R:
            for t in D:
                for i in N_C:
                    for j in N_C:
                        if i != j and (i, j) in A:
                            self.model.addConstr(self.W[i, t] + st[i, t] + tt[i, j] <= self.W[j, t] + M * (1 - self.X[i, j, t, r]), name=f"BigM_{i}_{j}_{r}_{t}")

        for i in N_TW:
            for t in D:
                self.model.addConstr(self.W[i, t] >= a[i], name=f"TW_Inicio_{i}_{t}")
                self.model.addConstr(self.W[i, t] <= b[i], name=f"TW_Fin_{i}_{t}")

        n_plus_1 = max(N)
        for j in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(self.dW[r] + tt[0, j] <= self.W[j, t] + M * (1 - self.X[0, j, t, r]), name=f"CoordSalida_{j}_{r}_{t}")
        for i in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(self.W[i, t] + st[i, t] + tt[i, n_plus_1] <= self.rW[r, t] + M * (1 - self.X[i, n_plus_1, t, r]), name=f"CoordRet_{i}_{r}_{t}")

        for r in R:
            self.model.addConstr(self.dW[r] >= a_W, name=f"Ap_Dep_{r}")
            for t in D:
                self.model.addConstr(self.rW[r, t] <= b_W, name=f"Ci_Dep_{r}_{t}")
                self.model.addConstr(gp.quicksum(q[i, t] * self.Z[i, r] for i in N_C) <= Q, name=f"Capacidad_{r}_{t}")  

        self.model.update()

    def set_base_params(self):
        self.model.setParam('Threads', 1)
        self.model.setParam('TimeLimit', 10)
        self.model.setParam('OutputFlag', 0)

    def run_relaxation(self):
        self.set_base_params()
        relaxed = self.model.relax()
        relaxed.optimize()
        return relaxed.ObjVal if relaxed.Status == GRB.OPTIMAL else None

    def run_configuration(self, config_name, params_dict):
        self.model.reset() 
        self.set_base_params()
        for p, v in params_dict.items():
            self.model.setParam(p, v)
            
        start = time.time()
        self.model.optimize()
        t = time.time() - start
        
        if self.model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT] and self.model.SolCount > 0:
            return self.model.ObjVal, self.model.MIPGap * 100, t
        return None, None, t

    def run_sensitivity_analysis(self, filename="sensibilidad_gurobi.csv"):
        # Seguro: Verificar que tengamos una solución antes de intentar fijar variables
        if self.model.SolCount == 0:
            print("\n[!] Advertencia: No hay solución factible en memoria.")
            print("[!] Gurobi no puede hacer análisis de sensibilidad sin una solución. Aumenta el TimeLimit.")
            return

        fixed_model = self.model.fixed()
        fixed_model.setParam('QCPDual', 1) 
        fixed_model.setParam('OutputFlag', 0)
        fixed_model.optimize()

        if fixed_model.Status == GRB.OPTIMAL:
            with open(filename, mode='w', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow(["Restriccion", "Precio_Sombra_Pi", "SARHSLow", "SARHSUp"])
                for r in self.data['R']:
                    for t in self.data['D']:
                        constr = fixed_model.getConstrByName(f"Capacidad_{r}_{t}")
                        if constr:
                            writer.writerow([f"Cap_{r}_{t}", f"{constr.Pi:.4f}", f"{constr.SARHSLow:.2f}", f"{constr.SARHSUp:.2f}"])
            print(f"Análisis de sensibilidad guardado exitosamente en {filename}")

    def exportar_resultados_csv(self, z_LP, res_milp, filename="resultados_gurobi.csv"):
        with open(filename, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Configuracion", "Valor_Objetivo", "Gap_%", "Tiempo_s", "Brecha_Integralidad_%"])
            writer.writerow(["LP Relax", f"{z_LP:.4f}" if z_LP else "N/A", "-", "-", "-"])
            
            z_MILP_base = res_milp.get("Baseline", (None, None, None))[0]
            for config, (obj, gap, run_time) in res_milp.items():
                brecha = f"{((z_MILP_base - z_LP) / z_MILP_base * 100):.2f}" if config == "Baseline" and z_LP and z_MILP_base else "-"
                writer.writerow([config, f"{obj:.4f}" if obj else "N/A", f"{gap:.2f}" if gap else "-", f"{run_time:.2f}", brecha])

### 3. Construcción y Ejecución de Experimentos

In [7]:
# Inicializar y construir modelo
framework = ConsistentVRPFramework(data_instancia)
framework.build_model()

print("1. Calculando Relajación Continua...")
z_LP = framework.run_relaxation()
print(f"Valor LP: {z_LP:.2f}")

# Configuraciones a testear
configs = {
    "Baseline": {},
    "No_Cuts": {'Cuts': 0},
    "No_Presolve": {'Presolve': 0},
    "MIPFocus_1": {'MIPFocus': 1},
    "MIPFocus_3": {'MIPFocus': 3},
    "No_Heuristics": {'Heuristics': 0}
}

resultados_milp = {}
print("\n2. Ejecutando Configuraciones MILP...")
for nombre, params in configs.items():
    print(f"Corriendo {nombre}...")
    obj, gap, t = framework.run_configuration(nombre, params)
    resultados_milp[nombre] = (obj, gap, t)
    print(f" -> Obj: {obj}, Gap: {gap}%, Tiempo: {t:.2f}s")

# Analisis de Sensibilidad y Exportación
framework.run_configuration("Baseline", {}) # Restaurar modelo a óptimo
framework.run_sensitivity_analysis("sensibilidad_gurobi.csv")
framework.exportar_resultados_csv(z_LP, resultados_milp, "resultados_gurobi.csv")
print("\n¡Experimentos finalizados y exportados a CSV!")

1. Calculando Relajación Continua...
Set parameter Threads to value 1
Set parameter TimeLimit to value 10
Valor LP: 1356.06

2. Ejecutando Configuraciones MILP...
Corriendo Baseline...
 -> Obj: 1546.7919972168436, Gap: 8.911127212783398%, Tiempo: 10.01s
Corriendo No_Cuts...
 -> Obj: None, Gap: None%, Tiempo: 10.00s
Corriendo No_Presolve...
 -> Obj: 1568.0895943654918, Gap: 12.709052678550293%, Tiempo: 10.01s
Corriendo MIPFocus_1...
 -> Obj: 1575.6598930272053, Gap: 13.821353132529154%, Tiempo: 10.01s
Corriendo MIPFocus_3...
 -> Obj: 1546.7919972168436, Gap: 9.652207941578638%, Tiempo: 10.00s
Corriendo No_Heuristics...
 -> Obj: None, Gap: None%, Tiempo: 10.00s

[!] Advertencia: No hay solución factible en memoria.
[!] Gurobi no puede hacer análisis de sensibilidad sin una solución. Aumenta el TimeLimit.

¡Experimentos finalizados y exportados a CSV!


### 4. Visualización de resultados

In [ ]:
import pandas as pd

print("--- Resultados de Configuraciones ---")
df_resultados = pd.read_csv("resultados_gurobi.csv")
display(df_resultados)

print("\n--- Análisis de Sensibilidad (Capacidad) ---")
df_sensibilidad = pd.read_csv("sensibilidad_gurobi.csv")
display(df_sensibilidad.head(6)) # Muestra las primeras 6 filas